In [5]:
# Import necessary libraries
import pandas as pd
import numpy as np
import requests
import json
import time

In [ ]:
# 50 Sites of Interest 
site_coords = {'KSGS.371852100505801': (37.31502, -100.8505),
 'KSGS.372043101363101': (37.34495, -101.6104),
 'KSGS.372539100142504': (37.42949, -100.2434),
 'KSGS.373331098033301': (37.56096, -98.058),
 'KSGS.373607100565301': (37.59874, -100.9497),
 'KSGS.374111099070401': (37.68448, -99.11925),
 'KSGS.374125100344101': (37.69024, -100.5795),
 'KSGS.374747100552101': (37.79492, -100.9225),
 'KSGS.375145100485701': (37.86209, -100.8179),
 'KSGS.375454101075401': (37.91572, -101.1306),
 'TWDB.0753701': (35.16, -102.4886),
 'TWDB.1005225': (34.97, -102.4381),
 'TWDB.1038101': (34.46, -102.3522),
 'TWDB.1039702': (34.38, -102.2336),
 'TWDB.1157901': (34.03, -101.9056),
 'TWDB.2409103': (33.8575, -102.9786),
 'TWDB.2409801': (33.76, -102.9564),
 'TWDB.2423701': (33.65222, -102.2278),
 'TWDB.2453902': (33.13833, -102.3808),
 'UNLCSD.423037099353801': (42.51028, -99.59429),
 'USGS.335258091152301': (33.88238, -91.25823),
 'USGS.340440091231201': (34.07647, -91.3878),
 'USGS.342649091251916': (34.46473, -91.42094),
 'USGS.345519091505401': (34.92182, -91.8482),
 'USGS.350029090265801': (35.0071, -90.44913),
 'USGS.373644113411501': (37.6122, -113.6883),
 'USGS.381901113014102': (38.31691, -113.0288),
 'USGS.383921112175901': (38.65806, -112.1672),
 'USGS.390819111530701': (39.13857, -111.886),
 'USGS.392601112412201': (39.43356, -112.6902),
 'USGS.400932101404101': (40.15888, -101.6785),
 'USGS.402103099402900': (40.35123, -99.67398),
 'USGS.402408099584800': (40.402, -99.98177),
 'USGS.403235101395501': (40.54472, -101.6668),
 'USGS.403949098182201': (40.66362, -98.30645),
 'USGS.412603099492001': (41.43417, -99.82262),
 'USGS.415754112551301': (41.96492, -112.9211),
 'USGS.433218112191601': (43.53824, -112.3219),
 'USGS.434102112180701': (43.68389, -112.3028),
 'USGS.434657112282201': (43.78211, -112.4735),
 'USSCS.352158090232301': (35.36662, -90.39071),
 'USSCS.355018090512401': (35.83535, -90.85817),
 'willcox': (32.03619, -109.754),
 'cuyama': (34.89025548, -119.6165171),
 'amargosa': (36.55634, -116.496),
 'san_simon_valley': (32.17119, -109.158),
 'butler_valley': (33.95808, -113.784),
 'mcmullen_valley': (33.95725, -113.08),
 'hualapai_valley': (35.47122, -113.976),
 'southern_willcox': (31.36597, -109.663)}

In [ ]:

# Insert OpenET API Key
header = {"Authorization": "API_KEY_GOES_HERE"}

sites = site_coords

# Select boundary box side length
size_km = 2

# Extract data sections less than 10 years as required by the API
date_ranges = [
    ("2000-01-01", "2009-12-31", 2.0), 
    ("2010-01-01", "2019-12-31", 2.0),
    ("2020-01-01", "2025-12-31", 2.1), # Use version 2.1 for newer versions which may not have accurate data for older versions
]

dfs = [] # Create empty dataframe

# Create function to generate 2km x 2km boundary box given coordinates from site_cords and boundary box size from size_km = 2

# "make_square_geometry" is Equivalent to the following for the first site from site_coords "KSGS.371852100505801" 
# Geometry follows OpenET API structure requirement so I dont think error is due to a geometry issue
# It forms a closed box using 5 latitude and longitude points

 # [-100.86182760428409,
 #37.324029009009,
 #-100.86182760428409,
 #37.30601099099099,
 #-100.8391723957159,
 #37.30601099099099,
 #-100.8391723957159,
 #37.324029009009,
 #-100.86182760428409,
 #37.324029009009]

def make_square_geometry(lat, lon, size_km):
    half = size_km / 2
    lat_offset = half / 111.0
    lon_offset = half / (111.0 * np.cos(np.radians(lat)))
    return [
        lon - lon_offset, lat + lat_offset,
        lon - lon_offset, lat - lat_offset,
        lon + lon_offset, lat - lat_offset,
        lon + lon_offset, lat + lat_offset,
        lon - lon_offset, lat + lat_offset,
    ]

# Function to retry failed requests to handle potential API rate limit errors
def query_with_retry(header, polygon_args, retries=3, wait=5):
    for attempt in range(retries):
        resp = requests.post(
            headers=header,
            json=polygon_args,
            url="https://openet-api.org/raster/timeseries/polygon"
        )
        if resp.status_code == 200:
            return resp
        print(f"    Attempt {attempt + 1} failed — retrying in {wait}s...")
        time.sleep(wait)
    return resp

# Use API to extract ET data 
for name, (lat, lon) in sites.items():
    print(f"Querying {name}...")
    
    for start_date, end_date, version in date_ranges:

        polygon_args = { # Define query parameters
            "date_range": [start_date, end_date],
            "interval": "monthly",
            "geometry": make_square_geometry(lat, lon, size_km),
            "model": "Ensemble",
            "reducer": "mean",
            "variable": "ET",
            "reference_et": "gridMET",
            "units": "mm",
            "file_format": "JSON",
            "version": version
        }

        resp = query_with_retry(header, polygon_args) # Request polygon API with retry logic

        # A single loop is equivalent to the following query request for the "Raster API"
        # This is an example for the first site from site_coords for the first date range of ("2000-01-01", "2009-12-31", 2.0), 
        # {
            #"date_range": [
                #"2000-01-01",
                #"2009-12-31"
            #],
            #"file_format": "JSON",
            #"geometry": [-100.86182760428409,
                        #37.324029009009,
                        #-100.86182760428409,
                        #37.30601099099099,
                        #-100.8391723957159,
                        #37.30601099099099,
                        #-100.8391723957159,
                        #37.324029009009,
                        #-100.86182760428409,
                        #37.324029009009
            #],
            #"interval": "monthly",
            #"model": "Ensemble", # The default API uses SSEBop but OpenET documentation claims ensemble is more accurate
            #"reducer": "mean",
            #"reference_et": "gridMET",
            #"units": "mm",
            #"variable": "ET",
            #"version": 2.0
        #}

        print(f"  {name} {start_date}–{end_date} — Status: {resp.status_code}") # Report status of data to console

        if resp.status_code != 200: # Return error given by API if unable to extract data
            print(f"  ERROR: {resp.json()}")
            continue

        data = resp.json()
        df = pd.DataFrame(data)

        # Append each query to a data frame
        dfs.append(df)

# Concatenate each appended query into a single data frame 
df_monthly = pd.concat(dfs, ignore_index=True) # Add new data as new observations in dataframe
print(df_monthly.head())